# The Potential of N-body Simulations

galpy can set up and work with the frozen potential of an N-body simulation. This allows you to study the properties of such potentials, investigate orbits, and calculate action-angle coordinates using the galpy framework. Currently, this functionality uses axisymmetrized versions of the snapshots. The use of this functionality requires [pynbody](https://github.com/pynbody/pynbody).

In [1]:
%matplotlib inline
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import numpy

## Simple example: SnapshotRZPotential

As a first, simple example we look at the potential of a single simulation particle, which should correspond to galpy's `KeplerPotential`. We can create such a single-particle snapshot using `pynbody` and then get its potential in galpy:

```python
import pynbody
s = pynbody.new(star=1)
s['mass'] = 1.
s['eps'] = 0.

from galpy.potential import SnapshotRZPotential
sp = SnapshotRZPotential(s, num_threads=1)
```

With these definitions, this snapshot potential should be the same as `KeplerPotential` with an amplitude of one, which we can verify:

```python
from galpy.potential import KeplerPotential
kp = KeplerPotential(amp=1.)
print(sp(1.1, 0.), kp(1.1, 0.), sp(1.1, 0.) - kp(1.1, 0.))
# (-0.90909090909090906, -0.9090909090909091, 0.0)
print(sp.Rforce(1.1, 0.), kp.Rforce(1.1, 0.), sp.Rforce(1.1, 0.) - kp.Rforce(1.1, 0.))
# (-0.82644628099173545, -0.8264462809917353, -1.1102230246251565e-16)
```

`SnapshotRZPotential` instances can be used wherever other galpy potentials can be used (note that the second derivatives have not been implemented, such that functions depending on those will not work). For example, we can plot the rotation curve:

```python
sp.plotRotcurve()
```

## InterpSnapshotRZPotential

Because evaluating the potential and forces of a snapshot is computationally expensive, most useful applications of frozen N-body potentials employ interpolated versions of the snapshot potential. These can be set up in galpy using `InterpSnapshotRZPotential`.

To illustrate its use we will make use of one of `pynbody`'s example snapshots, `g15784`. This snapshot is used in the [pynbody quickstart tutorial](https://pynbody.readthedocs.io/latest/tutorials/quickstart.html) and can be downloaded from [Zenodo](https://zenodo.org/records/12687409).

Once you have downloaded the testdata, load and prepare the snapshot:

```python
import pynbody
s = pynbody.load('g15784.lr.01024.gz')

# Get the main galaxy, center, and align face-on
h = s.halos()
h0 = h[0]
pynbody.analysis.faceon(h0)

# Convert to physical units with G=1
s.physical_units(mass='kpc km**2 s**-2 G**-1')
```

We can now load an interpolated version of this snapshot's potential into galpy:

```python
from galpy.potential import InterpSnapshotRZPotential
spi = InterpSnapshotRZPotential(
    h0,
    rgrid=(numpy.log(0.01), numpy.log(20.), 101),
    logR=True,
    zgrid=(0., 10., 101),
    interpPot=True,
    zsym=True,
)
```

where we further assume that the potential is symmetric around the mid-plane ($z=0$). This instantiation will take about ten to fifteen minutes. This potential instance has *physical* units (and thus the `rgrid=` and `zgrid=` inputs are given in kpc if the simulation's distance unit is kpc). For example, if we ask for the rotation curve:

```python
spi.plotRotcurve(
    Rrange=[0.01, 19.9],
    xlabel=r'$R\,(\mathrm{kpc})$',
    ylabel=r'$v_c(R)\,(\mathrm{km\,s}^{-1})$',
)
```

This can be compared to the rotation curve calculated via direct force summation by `pynbody`, see [here](https://pynbody.readthedocs.io/latest/tutorials/profile.html#rotation-curves).

## Converting to natural units

Because galpy works best in a system of *natural units*, we can convert the interpolated snapshot potential to natural units using `normalize()`. For example, using the circular velocity at $R = 10$ kpc:

```python
spi.vcirc(10.)
# 294.62723076942245

spi.normalize(R0=10.)
```

We can then plot the rotation curve again, keeping in mind that the distance unit is now $R_0$:

```python
spi.plotRotcurve(Rrange=[0.01, 1.99])
```

In particular, `spi.vcirc(1.)` now returns `1.0`. We can also plot the potential:

```python
spi.plot(rmin=0.01, rmax=1.9, nrs=51, zmin=-0.99, zmax=0.99, nzs=51)
```

This simulation's potential is quite spherical, which is confirmed by looking at the flattening:

```python
spi.flattening(1., 0.1)
# 0.86675711023391921
spi.flattening(1.5, 0.1)
# 0.94442750306256895
```

The epicycle and vertical frequencies can also be interpolated by setting the `interpepifreq=True` or `interpverticalfreq=True` keywords when instantiating the `InterpSnapshotRZPotential` object.